<a href="https://colab.research.google.com/github/Gibsdevops/Large-language-models/blob/main/Building_GPT_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

BUILDING GPT FROM SCARTCH USING THIS BOOK

https://knowledge.zhaoweiguo.com/build/html/x-learning/books/ais/2024/build_llm_from_scratch#understanding-llm



## CHAPTER TWO

In [1]:
text = "The cat sat on the mat because the cat was tired."

print(text)
print("Number of characters:", len(text))

The cat sat on the mat because the cat was tired.
Number of characters: 49


In [2]:
#At this point GPT cannot process this string directly. We need the Chapter 2 pipeline:
#WE NEED TO TRANSFORM; RAW TEXT --> TOKENS --> TOKEN_IDS --> INPUT SREQUENCE

#The book specifically uses GPT-2's BPE tokenizer via tiktoken;
#it explains that BPE can break text into subword units and then later uses sliding windows to create input/target pairs.

!pip -q install tiktoken

In [3]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

token_ids = tokenizer.encode(text)

print("Token IDs:")
print(token_ids)

print("\nNumber of tokens:", len(token_ids))

Token IDs:
[464, 3797, 3332, 319, 262, 2603, 780, 262, 3797, 373, 10032, 13]

Number of tokens: 12


In [4]:
#Look inside the tokenization
for token_id in token_ids:
    token = tokenizer.decode([token_id])
    print(f"{repr(token):12} -> {token_id}")

'The'        -> 464
' cat'       -> 3797
' sat'       -> 3332
' on'        -> 319
' the'       -> 262
' mat'       -> 2603
' because'   -> 780
' the'       -> 262
' cat'       -> 3797
' was'       -> 373
' tired'     -> 10032
'.'          -> 13


Notice something interesting when you run it: GPT-2's tokenizer often includes the preceding space as part of the token, so you may see ' cat' rather than 'cat'.

In [5]:
#Create one input/target pair
#suppose we choose context length of 4
#eg [464, 3797, 3332, 319, 262] these would be the first five tokens
#why 5
# INPUT:   The   cat   sat   on
# TARGET:  cat   sat   on    the

#we need this pair the extract one is the match of the forth toke

context_length = 4
chunk = token_ids[:context_length + 1]

input_ids = chunk[:-1]
target_ids = chunk[1:]

print("Chunk:  ", chunk)
print("Input:  ", input_ids)
print("Target: ", target_ids)

print("\nInput text: ", tokenizer.decode(input_ids))
print("Target text:", tokenizer.decode(target_ids))


Chunk:   [464, 3797, 3332, 319, 262]
Input:   [464, 3797, 3332, 319]
Target:  [3797, 3332, 319, 262]

Input text:  The cat sat on
Target text:  cat sat on the


In [6]:
# sliding window
#we know the context length
#use use a stride here eg 4
#this defines how many token positions should the window move after creating one training example


stride = 4

for i in range(0, len(token_ids) - context_length, stride):

    input_chunk = token_ids[i : i + context_length]
    target_chunk = token_ids[i + 1 : i + context_length + 1]

    print("Start position:", i)
    print("Input: ", tokenizer.decode(input_chunk))
    print("Target:", tokenizer.decode(target_chunk))
    print()

Start position: 0
Input:  The cat sat on
Target:  cat sat on the

Start position: 4
Input:   the mat because the
Target:  mat because the cat



In [7]:
#why is it called a sliding window
#change the stride to 1
context_length = 4
stride = 1

for i in range(0, len(token_ids) - context_length, stride):

    input_chunk = token_ids[i : i + context_length]
    target_chunk = token_ids[i + 1 : i + context_length + 1]

    print("Start position:", i)
    print("Input: ", tokenizer.decode(input_chunk))
    print("Target:", tokenizer.decode(target_chunk))
    print()

Start position: 0
Input:  The cat sat on
Target:  cat sat on the

Start position: 1
Input:   cat sat on the
Target:  sat on the mat

Start position: 2
Input:   sat on the mat
Target:  on the mat because

Start position: 3
Input:   on the mat because
Target:  the mat because the

Start position: 4
Input:   the mat because the
Target:  mat because the cat

Start position: 5
Input:   mat because the cat
Target:  because the cat was

Start position: 6
Input:   because the cat was
Target:  the cat was tired

Start position: 7
Input:   the cat was tired
Target:  cat was tired.



In [8]:
#Next: turn this into a PyTorch dataset
import torch
from torch.utils.data import Dataset


class GPTDataset(Dataset):

    #building the training examples
    def __init__(self, token_ids, context_length, stride):

        self.input_ids = []
        self.target_ids = []

        for i in range(0, len(token_ids) - context_length, stride):

            input_chunk = token_ids[i : i + context_length]
            target_chunk = token_ids[i + 1 : i + context_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    #how many examples do we have
    def __len__(self):
        return len(self.input_ids)

    #give me an example X
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

In [9]:
#we use it
dataset = GPTDataset(
    token_ids,
    context_length=4,
    stride=1
)

print("Number of training examples:", len(dataset))

input_ids, target_ids = dataset[0]

print("\nFirst example:")
print("Input IDs: ", input_ids)
print("Target IDs:", target_ids)

print("\nInput text: ", tokenizer.decode(input_ids.tolist()))
print("Target text:", tokenizer.decode(target_ids.tolist()))

Number of training examples: 8

First example:
Input IDs:  tensor([ 464, 3797, 3332,  319])
Target IDs: tensor([3797, 3332,  319,  262])

Input text:  The cat sat on
Target text:  cat sat on the


In [10]:
# DataLoader and batches
from torch.utils.data import DataLoader

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False
)

inputs, targets = next(iter(dataloader))

print("Inputs:")
print(inputs)

print("\nTargets:")
print(targets)

print("\nInput shape:", inputs.shape)
print("Target shape:", targets.shape)

Inputs:
tensor([[ 464, 3797, 3332,  319],
        [3797, 3332,  319,  262]])

Targets:
tensor([[3797, 3332,  319,  262],
        [3332,  319,  262, 2603]])

Input shape: torch.Size([2, 4])
Target shape: torch.Size([2, 4])


In [11]:
#turn token IDs into embeddings
#the shape of both the input and target pairs is [2, 4]

#but Attention cannot meaningfully operate directly on IDs such as 464 or 3797.

#turn each IDs into an embedding
#we define an embedding dimension
import torch.nn as nn

vocab_size = tokenizer.n_vocab
embedding_dim = 8

token_embedding = nn.Embedding(
    vocab_size,
    embedding_dim
)

input_embeddings = token_embedding(inputs)

print("Vocabulary size:", vocab_size)
print("Input IDs shape:", inputs.shape)
print("Embeddings shape:", input_embeddings.shape)

Vocabulary size: 50257
Input IDs shape: torch.Size([2, 4])
Embeddings shape: torch.Size([2, 4, 8])


In [12]:
#create positional embeddings
context_length = 4

position_embedding = nn.Embedding(
    context_length,
    embedding_dim
)

positions = torch.arange(context_length)

pos_embeddings = position_embedding(positions)

print("Positions:")
print(positions)

print("\nPosition embeddings shape:")
print(pos_embeddings.shape)

print("\nPosition embeddings:")
print(pos_embeddings)

Positions:
tensor([0, 1, 2, 3])

Position embeddings shape:
torch.Size([4, 8])

Position embeddings:
tensor([[-0.7823,  0.8733,  0.4647,  0.6717,  1.3942, -0.5504, -0.9854,  1.1687],
        [ 1.2126, -0.3932, -0.8497,  0.1356,  0.8006,  0.3102, -0.6688,  0.7653],
        [-0.8659, -0.3540,  0.6312,  1.7460, -2.3040, -1.6120, -1.6523, -1.1860],
        [ 2.1224, -0.5543,  0.3080,  0.0615,  0.2336, -0.6926,  1.3348, -0.0043]],
       grad_fn=<EmbeddingBackward0>)


In [13]:
#Combine token + position embeddings
input_embeddings = token_embedding(inputs)

gpt_inputs = input_embeddings + pos_embeddings

print("Token embeddings shape:")
print(input_embeddings.shape)

print("\nPosition embeddings shape:")
print(pos_embeddings.shape)

print("\nFinal GPT input shape:")
print(gpt_inputs.shape)

Token embeddings shape:
torch.Size([2, 4, 8])

Position embeddings shape:
torch.Size([4, 8])

Final GPT input shape:
torch.Size([2, 4, 8])


## CHAPTER THREE ATTENTION MECHANISMS

In [14]:
#For learning purposes, let's remove the batch dimension temporarily and work with the first sequence:
# we use "The cat sat on"


In [15]:
x = gpt_inputs[0]

print("Shape of x:", x.shape)
print("\nx:")
print(x)

Shape of x: torch.Size([4, 8])

x:
tensor([[ 0.8550,  1.5736,  0.8295,  0.2192,  0.3676, -2.3028, -2.8853,  0.8952],
        [ 1.6010, -0.8681, -0.2908, -0.0989,  1.4013, -0.9785, -1.7031,  0.3130],
        [-1.6379, -0.6779,  0.1443,  1.8830, -1.6142, -1.0659, -1.9434, -0.7895],
        [ 1.8028, -1.6682,  0.9041, -0.2500, -1.0969, -0.2617,  2.2453, -0.1110]],
       grad_fn=<SelectBackward0>)


In [16]:
#Each row is now the representation of one token including both token and positional information.
#now we will transform this X into the Q, K, V

#WE create the Q, K, V step
#CREATE THE Wq, Wk, Wv
#we decide to have our q, k, v 4 dimensional
#so we need a transformation that takes in 8 and gives 4
d_in = 8
d_out = 4

#query weights
W_query = nn.Linear(d_in, d_out, bias=False)
#key weights
W_key   = nn.Linear(d_in, d_out, bias=False)
#value weights
W_value = nn.Linear(d_in, d_out, bias=False)

print("W_query weight shape:", W_query.weight.shape)
print("W_key weight shape:  ", W_key.weight.shape)
print("W_value weight shape:", W_value.weight.shape)

W_query weight shape: torch.Size([4, 8])
W_key weight shape:   torch.Size([4, 8])
W_value weight shape: torch.Size([4, 8])


In [17]:
#it has printed torch.Size([4, 8]) because here pytorch represents 4 as output, 8 as input


In [18]:
#Calculate Q, K, V
Q = W_query(x)
K = W_key(x)
V = W_value(x)

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

print("\nQ:")
print(Q)

print("\nK:")
print(K)

print("\nV:")
print(V)

Q shape: torch.Size([4, 4])
K shape: torch.Size([4, 4])
V shape: torch.Size([4, 4])

Q:
tensor([[-0.1475, -0.0739, -1.0481,  1.1277],
        [-0.5507, -0.6256,  0.1524, -0.1282],
        [-0.3144,  0.0915, -1.8162,  0.4624],
        [-0.3820, -0.0325,  0.3707, -0.3063]], grad_fn=<MmBackward0>)

K:
tensor([[-0.7089, -0.5080, -1.2705, -0.0929],
        [-0.3855,  0.4702, -0.1909,  0.1558],
        [ 0.9710,  0.0291, -1.0532,  0.1379],
        [ 0.8371,  0.6961,  0.7760, -0.7935]], grad_fn=<MmBackward0>)

V:
tensor([[ 0.9630, -1.0090,  0.2813, -0.4390],
        [-0.0945, -0.3395,  0.2292, -0.4191],
        [ 1.2448, -1.1914, -0.4079, -0.5245],
        [-1.1786, -0.0755,  0.1104, -0.2438]], grad_fn=<MmBackward0>)


In [19]:
# #Compute attention scores
# Remember the intuition:
# - Query = what am I looking for?
# - Key = what information might I be relevant for?
# - Value = what information do I actually contribute?
# For every token, we compare its query with every token's key using a dot product.

attention_scores = Q @ K.T

print("Attention scores shape:", attention_scores.shape)

print("\nAttention scores:")
print(attention_scores)

Attention scores shape: torch.Size([4, 4])

Attention scores:
tensor([[ 1.3690,  0.3979,  1.1139, -1.8831],
        [ 0.5265, -0.1309, -0.7310, -0.6765],
        [ 2.4408,  0.5830,  1.6740, -1.9757],
        [-0.1552,  0.0135, -0.8045,  0.1884]], grad_fn=<MmBackward0>)


In [20]:
#Scale the scores
import math

d_k = K.shape[-1]

scaled_scores = attention_scores / math.sqrt(d_k)

print("d_k:", d_k)
print("sqrt(d_k):", math.sqrt(d_k))

print("\nScaled attention scores:")
print(scaled_scores)

d_k: 4
sqrt(d_k): 2.0

Scaled attention scores:
tensor([[ 0.6845,  0.1990,  0.5570, -0.9415],
        [ 0.2633, -0.0654, -0.3655, -0.3382],
        [ 1.2204,  0.2915,  0.8370, -0.9879],
        [-0.0776,  0.0067, -0.4023,  0.0942]], grad_fn=<DivBackward0>)


In [21]:
# # Build the causal mask
# The → The

# cat → The, cat

# sat → The, cat, sat

# on  → The, cat, sat, on


In [22]:
#create the mask without applying it
seq_len = scaled_scores.shape[0]

mask = torch.triu(
    torch.ones(seq_len, seq_len),
    diagonal=1
)

print("Sequence length:", seq_len)
print("\nCausal mask:")
print(mask)

Sequence length: 4

Causal mask:
tensor([[0., 1., 1., 1.],
        [0., 0., 1., 1.],
        [0., 0., 0., 1.],
        [0., 0., 0., 0.]])


In [23]:
#after creating the attention matrix we now apply that matrix to the attention scores
#the scores that we scaled
#we want every position where mask == 1 to become - infintity
masked_scores = scaled_scores.masked_fill(
    mask.bool(),
    float("-inf")
)

print("Masked attention scores:")
print(masked_scores)

#mask.bool() converts 0 false and 1 to true,
#masked_fill() says where there's True but -inf

Masked attention scores:
tensor([[ 0.6845,    -inf,    -inf,    -inf],
        [ 0.2633, -0.0654,    -inf,    -inf],
        [ 1.2204,  0.2915,  0.8370,    -inf],
        [-0.0776,  0.0067, -0.4023,  0.0942]], grad_fn=<MaskedFillBackward0>)


Notice something important: we haven't deleted the future tokens from the sequence. They are still present in the tensors. We've simply made their attention scores impossible to select.

In [24]:
#Softmax
attention_weights = torch.softmax(
    masked_scores,
    dim=-1
)

print("Attention weights:")
print(attention_weights)

print("\nEach row sums to:")
print(attention_weights.sum(dim=-1))

Attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5814, 0.4186, 0.0000, 0.0000],
        [0.4816, 0.1902, 0.3282, 0.0000],
        [0.2501, 0.2721, 0.1808, 0.2970]], grad_fn=<SoftmaxBackward0>)

Each row sums to:
tensor([1., 1., 1., 1.], grad_fn=<SumBackward1>)


In [25]:
# #here dim = -1 means apply softmax across the columns of each row
# That's exactly what we want because each row represents one query asking:
# How much attention should I give to each key?

In [26]:
#use the values
context_vectors = attention_weights @ V

print("Context vectors shape:")
print(context_vectors.shape)

print("\nContext vectors:")
print(context_vectors)

Context vectors shape:
torch.Size([4, 4])

Context vectors:
tensor([[ 0.9630, -1.0090,  0.2813, -0.4390],
        [ 0.5203, -0.7288,  0.2595, -0.4306],
        [ 0.8543, -0.9415,  0.0452, -0.4633],
        [ 0.0901, -0.5825,  0.0918, -0.3910]], grad_fn=<MmBackward0>)


In [27]:
#We have now completed one full causal self-attention head. Before multi-head attention, it's worth seeing what your output actually means.


Now we move into multi-head attention. The key idea is simple: we just built one attention head. Multi-head attention runs several versions of that same process in parallel. The book moves from causal attention into multi-head attention at this stage.

In [28]:
#create two independent heads
head_dim = 4 #each head has 4 dimensions

# Head 1
W_query_1 = nn.Linear(8, head_dim, bias=False)
W_key_1   = nn.Linear(8, head_dim, bias=False)
W_value_1 = nn.Linear(8, head_dim, bias=False)

# Head 2
W_query_2 = nn.Linear(8, head_dim, bias=False)
W_key_2   = nn.Linear(8, head_dim, bias=False)
W_value_2 = nn.Linear(8, head_dim, bias=False)

print("Head 1 query weight shape:", W_query_1.weight.shape)
print("Head 2 query weight shape:", W_query_2.weight.shape)

Head 1 query weight shape: torch.Size([4, 8])
Head 2 query weight shape: torch.Size([4, 8])


Both heads have the same architecture, but because we created separate nn.Linear layers, they contain different randomly initialized learnable parameters.

In [29]:
#let's pass the same X through the two heads
#to get the Q, K and V of the two different heads
#so we shall get Q1, Q2, K1, K2, V1, V2

# Head 1
Q1 = W_query_1(x)
K1 = W_key_1(x)
V1 = W_value_1(x)

# Head 2
Q2 = W_query_2(x)
K2 = W_key_2(x)
V2 = W_value_2(x)

print("Input x shape:", x.shape)

print("\nHead 1:")
print("Q1 shape:", Q1.shape)
print("K1 shape:", K1.shape)
print("V1 shape:", V1.shape)

print("\nHead 2:")
print("Q2 shape:", Q2.shape)
print("K2 shape:", K2.shape)
print("V2 shape:", V2.shape)

print("\nQ1:")
print(Q1)

print("\nQ2:")
print(Q2)

Input x shape: torch.Size([4, 8])

Head 1:
Q1 shape: torch.Size([4, 4])
K1 shape: torch.Size([4, 4])
V1 shape: torch.Size([4, 4])

Head 2:
Q2 shape: torch.Size([4, 4])
K2 shape: torch.Size([4, 4])
V2 shape: torch.Size([4, 4])

Q1:
tensor([[-0.8347, -0.7125, -1.0488, -0.4971],
        [-1.8609, -0.0221,  0.3496,  0.0989],
        [-0.2011,  0.4460, -1.2753, -0.3047],
        [ 0.1805,  0.9979,  0.8951, -0.4796]], grad_fn=<MmBackward0>)

Q2:
tensor([[ 0.1129,  0.8259,  0.7694,  0.5270],
        [-0.3919, -0.1050,  0.6322,  0.2448],
        [-1.8623,  0.4198,  1.0269,  1.7314],
        [ 0.6962, -1.0512, -0.1077, -0.9055]], grad_fn=<MmBackward0>)


In [30]:
# Raw attention scores
scores_1 = Q1 @ K1.T
scores_2 = Q2 @ K2.T

# Scale
scores_1 = scores_1 / math.sqrt(head_dim)
scores_2 = scores_2 / math.sqrt(head_dim)

# Apply the SAME causal mask to both heads
scores_1 = scores_1.masked_fill(mask.bool(), float("-inf"))
scores_2 = scores_2.masked_fill(mask.bool(), float("-inf"))

# Convert to attention weights
attn_weights_1 = torch.softmax(scores_1, dim=-1)
attn_weights_2 = torch.softmax(scores_2, dim=-1)

print("Head 1 attention weights:")
print(attn_weights_1)

print("\nHead 2 attention weights:")
print(attn_weights_2)

Head 1 attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5294, 0.4706, 0.0000, 0.0000],
        [0.3257, 0.5119, 0.1623, 0.0000],
        [0.3779, 0.1717, 0.3024, 0.1481]], grad_fn=<SoftmaxBackward0>)

Head 2 attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5094, 0.4906, 0.0000, 0.0000],
        [0.1478, 0.2182, 0.6340, 0.0000],
        [0.3546, 0.2667, 0.3085, 0.0702]], grad_fn=<SoftmaxBackward0>)


In [31]:
#Produce each head's context vectors
context_1 = attn_weights_1 @ V1
context_2 = attn_weights_2 @ V2

print("Head 1 context shape:", context_1.shape)
print("Head 2 context shape:", context_2.shape)

print("\nHead 1 context:")
print(context_1)

print("\nHead 2 context:")
print(context_2)

Head 1 context shape: torch.Size([4, 4])
Head 2 context shape: torch.Size([4, 4])

Head 1 context:
tensor([[ 0.6560,  0.4600, -1.4771, -0.4405],
        [ 0.2323,  0.7806, -0.7246, -0.3308],
        [-0.0400,  0.5843, -0.5054, -0.2965],
        [-0.0748,  0.2235, -0.7176, -0.2346]], grad_fn=<MmBackward0>)

Head 2 context:
tensor([[ 1.4601,  0.4579, -0.3940, -0.7910],
        [ 0.7325,  0.0267, -0.1663, -0.4094],
        [ 0.6406,  0.6661, -1.1953, -0.8001],
        [ 0.6329,  0.3542, -0.6510, -0.5681]], grad_fn=<MmBackward0>)


In [32]:
#Concatenate the heads
multi_head_output = torch.cat(
    [context_1, context_2],
    dim=-1
)

print("Head 1 shape:", context_1.shape)
print("Head 2 shape:", context_2.shape)
print("Combined shape:", multi_head_output.shape)

print("\nCombined multi-head output:")
print(multi_head_output)

Head 1 shape: torch.Size([4, 4])
Head 2 shape: torch.Size([4, 4])
Combined shape: torch.Size([4, 8])

Combined multi-head output:
tensor([[ 0.6560,  0.4600, -1.4771, -0.4405,  1.4601,  0.4579, -0.3940, -0.7910],
        [ 0.2323,  0.7806, -0.7246, -0.3308,  0.7325,  0.0267, -0.1663, -0.4094],
        [-0.0400,  0.5843, -0.5054, -0.2965,  0.6406,  0.6661, -1.1953, -0.8001],
        [-0.0748,  0.2235, -0.7176, -0.2346,  0.6329,  0.3542, -0.6510, -0.5681]],
       grad_fn=<CatBackward0>)


Output projection Wo

Right now our vector is literally a concatenation:
We don't want those head outputs to remain as two isolated chunks forever. We want the model to learn how to combine information across the heads.

In [33]:
#that's the job of output project
#create  one more linear layer

output_projection = nn.Linear(8, 8, bias=False)

final_attention_output = output_projection(multi_head_output)

print("Before output projection:", multi_head_output.shape)
print("Output projection weight:", output_projection.weight.shape)
print("After output projection:", final_attention_output.shape)

print("\nFinal attention output:")
print(final_attention_output)

Before output projection: torch.Size([4, 8])
Output projection weight: torch.Size([8, 8])
After output projection: torch.Size([4, 8])

Final attention output:
tensor([[-0.0572,  0.7152, -0.0619, -0.1668,  0.6809,  0.1422, -0.1261, -0.5277],
        [-0.2693,  0.2555, -0.2349, -0.0682,  0.3436,  0.1433, -0.2362, -0.1068],
        [-0.2374,  0.4417, -0.1497,  0.2017,  0.3614,  0.0450, -0.3522, -0.1035],
        [-0.0851,  0.4259, -0.0749,  0.0330,  0.3079,  0.0687, -0.2014, -0.2401]],
       grad_fn=<MmBackward0>)


In [34]:
# Start a MultiHeadAttention class
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads):
        super().__init__()

        assert d_out % num_heads == 0

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

        self.out_proj = nn.Linear(d_out, d_out, bias=False)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )


In [35]:
#Instantiate our Multi-Head Attention

mha = MultiHeadAttention(
    d_in=8,
    d_out=8,
    context_length=4,
    num_heads=2
)

print("Number of heads:", mha.num_heads)
print("Head dimension:", mha.head_dim)

print("\nW_query weight shape:", mha.W_query.weight.shape)
print("W_key weight shape:", mha.W_key.weight.shape)
print("W_value weight shape:", mha.W_value.weight.shape)
print("Output projection shape:", mha.out_proj.weight.shape)

print("\nCausal mask:")
print(mha.mask)

Number of heads: 2
Head dimension: 4

W_query weight shape: torch.Size([8, 8])
W_key weight shape: torch.Size([8, 8])
W_value weight shape: torch.Size([8, 8])
Output projection shape: torch.Size([8, 8])

Causal mask:
tensor([[0., 1., 1., 1.],
        [0., 0., 1., 1.],
        [0., 0., 0., 1.],
        [0., 0., 0., 0.]])


In [36]:
#Project Q, K, V and split into heads
x_batch = gpt_inputs

batch_size, num_tokens, d_in = x_batch.shape

# Project input into Q, K, V
queries = mha.W_query(x_batch)
keys    = mha.W_key(x_batch)
values  = mha.W_value(x_batch)

print("Input shape:", x_batch.shape)
print("Queries shape:", queries.shape)
print("Keys shape:", keys.shape)
print("Values shape:", values.shape)

Input shape: torch.Size([2, 4, 8])
Queries shape: torch.Size([2, 4, 8])
Keys shape: torch.Size([2, 4, 8])
Values shape: torch.Size([2, 4, 8])


In [37]:
#Split Q, K, V into heads
queries_split = queries.view(
    batch_size,
    num_tokens,
    mha.num_heads,
    mha.head_dim
)

keys_split = keys.view(
    batch_size,
    num_tokens,
    mha.num_heads,
    mha.head_dim
)

values_split = values.view(
    batch_size,
    num_tokens,
    mha.num_heads,
    mha.head_dim
)

print("Before splitting:", queries.shape)
print("After splitting:", queries_split.shape)

Before splitting: torch.Size([2, 4, 8])
After splitting: torch.Size([2, 4, 2, 4])


[2, 4, 2, 4]

2 --> sequences

4 --> tokens

2 --> heads

4 --> features

In [38]:
#but heads must be between the Batch, and tokens
#[B, H, T, dhead]

#swap dimensions
queries_heads = queries_split.transpose(1, 2)
keys_heads    = keys_split.transpose(1, 2)
values_heads  = values_split.transpose(1, 2)

print("Before transpose:", queries_split.shape)
print("After transpose:", queries_heads.shape)

print("\nKeys shape:", keys_heads.shape)
print("Values shape:", values_heads.shape)

Before transpose: torch.Size([2, 4, 2, 4])
After transpose: torch.Size([2, 2, 4, 4])

Keys shape: torch.Size([2, 2, 4, 4])
Values shape: torch.Size([2, 2, 4, 4])


In [39]:
#Calculate all attention scores at once
attention_scores = queries_heads @ keys_heads.transpose(-2, -1)

print("Queries shape:", queries_heads.shape)
print("Transposed keys shape:", keys_heads.transpose(-2, -1).shape)
print("Attention scores shape:", attention_scores.shape)

Queries shape: torch.Size([2, 2, 4, 4])
Transposed keys shape: torch.Size([2, 2, 4, 4])
Attention scores shape: torch.Size([2, 2, 4, 4])


In [40]:
#Inspect one head from one batch
print("Batch 0, Head 0:")
print(attention_scores[0, 0])

print("\nShape:")
print(attention_scores[0, 0].shape)

Batch 0, Head 0:
tensor([[ 0.1528,  0.1593, -2.2076,  0.4786],
        [ 0.6490,  0.0075, -0.8293,  0.5359],
        [ 0.0339,  0.3553,  1.0621,  0.4733],
        [ 0.7177, -0.2628, -0.1569, -0.7323]], grad_fn=<SelectBackward0>)

Shape:
torch.Size([4, 4])


In [41]:
scaled_scores = attention_scores / math.sqrt(mha.head_dim)

print("Scaled scores shape:", scaled_scores.shape)

print("\nBatch 0, Head 0 after scaling:")
print(scaled_scores[0, 0])

Scaled scores shape: torch.Size([2, 2, 4, 4])

Batch 0, Head 0 after scaling:
tensor([[ 0.0764,  0.0797, -1.1038,  0.2393],
        [ 0.3245,  0.0037, -0.4146,  0.2679],
        [ 0.0169,  0.1776,  0.5310,  0.2366],
        [ 0.3588, -0.1314, -0.0785, -0.3661]], grad_fn=<SelectBackward0>)


In [42]:
#Apply the causal mask with broadcasting
masked_scores = scaled_scores.masked_fill(
    mha.mask.bool(),
    float("-inf")
)

print("Mask shape:", mha.mask.shape)
print("Scores shape:", scaled_scores.shape)
print("Masked scores shape:", masked_scores.shape)

print("\nBatch 0, Head 0 after masking:")
print(masked_scores[0, 0])

Mask shape: torch.Size([4, 4])
Scores shape: torch.Size([2, 2, 4, 4])
Masked scores shape: torch.Size([2, 2, 4, 4])

Batch 0, Head 0 after masking:
tensor([[ 0.0764,    -inf,    -inf,    -inf],
        [ 0.3245,  0.0037,    -inf,    -inf],
        [ 0.0169,  0.1776,  0.5310,    -inf],
        [ 0.3588, -0.1314, -0.0785, -0.3661]], grad_fn=<SelectBackward0>)


In [43]:
#Softmax across the key dimension
attention_weights = torch.softmax(
    masked_scores,
    dim=-1
)

print("Attention weights shape:", attention_weights.shape)

print("\nBatch 0, Head 0 attention weights:")
print(attention_weights[0, 0])

print("\nRow sums:")
print(attention_weights[0, 0].sum(dim=-1))

Attention weights shape: torch.Size([2, 2, 4, 4])

Batch 0, Head 0 attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5795, 0.4205, 0.0000, 0.0000],
        [0.2600, 0.3053, 0.4347, 0.0000],
        [0.3646, 0.2233, 0.2355, 0.1766]], grad_fn=<SelectBackward0>)

Row sums:
tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [44]:
#Multiply attention weights by Values
context = attention_weights @ values_heads

print("Attention weights shape:", attention_weights.shape)
print("Values shape:", values_heads.shape)
print("Context shape:", context.shape)

print("\nBatch 0, Head 0 context vectors:")
print(context[0, 0])

Attention weights shape: torch.Size([2, 2, 4, 4])
Values shape: torch.Size([2, 2, 4, 4])
Context shape: torch.Size([2, 2, 4, 4])

Batch 0, Head 0 context vectors:
tensor([[ 0.6813, -1.1431,  0.1482,  0.3981],
        [ 0.6428, -0.7000,  0.1308,  0.1404],
        [ 0.2761, -1.1547, -0.4120,  0.1424],
        [ 0.4515, -0.7722,  0.0897,  0.1282]], grad_fn=<SelectBackward0>)


In [45]:
#Concatenate the heads back together
context_transposed = context.transpose(1, 2)

context_combined = context_transposed.contiguous().view(
    batch_size,
    num_tokens,
    mha.d_out
)

print("Before transpose:", context.shape)
print("After transpose:", context_transposed.shape)
print("After combining heads:", context_combined.shape)

Before transpose: torch.Size([2, 2, 4, 4])
After transpose: torch.Size([2, 4, 2, 4])
After combining heads: torch.Size([2, 4, 8])


In [46]:
#apply projection Wo
final_output = mha.out_proj(context_combined)

print("Before output projection:", context_combined.shape)
print("W_O shape:", mha.out_proj.weight.shape)
print("Final output shape:", final_output.shape)

print("\nBatch 0 final output:")
print(final_output[0])

Before output projection: torch.Size([2, 4, 8])
W_O shape: torch.Size([8, 8])
Final output shape: torch.Size([2, 4, 8])

Batch 0 final output:
tensor([[-0.5575,  0.2712, -0.6400,  0.1391,  0.2644,  0.1464,  0.2510, -0.7321],
        [-0.3644,  0.2872, -0.4712,  0.1752,  0.1398, -0.0032,  0.3219, -0.4394],
        [-0.4171,  0.2252, -0.7515,  0.1144,  0.2590, -0.2036,  0.2734, -0.4709],
        [-0.3534,  0.1945, -0.4711,  0.1531,  0.1732, -0.0207,  0.2482, -0.4562]],
       grad_fn=<SelectBackward0>)


Complete the MultiHeadAttention class

In [47]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads):
        super().__init__()

        assert d_out % num_heads == 0

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        # Q, K, V projections
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

        # Output projection W_O
        self.out_proj = nn.Linear(d_out, d_out, bias=False)

        # Causal mask
        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )

    def forward(self, x):

        # x: [B, T, d_in]
        B, T, _ = x.shape

        # 1. Create Q, K, V
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        # [B, T, d_out]
        #       ↓
        # [B, T, H, head_dim]
        queries = queries.view(B, T, self.num_heads, self.head_dim)
        keys    = keys.view(B, T, self.num_heads, self.head_dim)
        values  = values.view(B, T, self.num_heads, self.head_dim)

        # [B, T, H, head_dim]
        #       ↓
        # [B, H, T, head_dim]
        queries = queries.transpose(1, 2)
        keys    = keys.transpose(1, 2)
        values  = values.transpose(1, 2)

        # 2. QK^T / sqrt(d_k)
        attention_scores = queries @ keys.transpose(-2, -1)
        attention_scores = attention_scores / math.sqrt(self.head_dim)

        # 3. Causal mask
        attention_scores = attention_scores.masked_fill(
            self.mask[:T, :T].bool(),
            float("-inf")
        )

        # 4. Softmax
        attention_weights = torch.softmax(
            attention_scores,
            dim=-1
        )

        # 5. Weighted sum of Values
        context = attention_weights @ values

        # [B, H, T, head_dim]
        #       ↓
        # [B, T, H, head_dim]
        context = context.transpose(1, 2)

        # concatenate heads
        # [B, T, H, head_dim]
        #       ↓
        # [B, T, d_out]
        context = context.contiguous().view(
            B, T, self.d_out
        )

        # 6. Output projection
        output = self.out_proj(context)

        return output

In [48]:
#Test our MultiHeadAttention
mha_test = MultiHeadAttention(
    d_in=8,
    d_out=8,
    context_length=4,
    num_heads=2
)

output = mha_test(gpt_inputs)

print("Input shape:", gpt_inputs.shape)
print("Output shape:", output.shape)

print("\nBatch 0 output:")
print(output[0])

Input shape: torch.Size([2, 4, 8])
Output shape: torch.Size([2, 4, 8])

Batch 0 output:
tensor([[ 0.6649, -0.2722,  0.2148,  1.3442,  0.4512, -0.0245,  0.0608,  0.7016],
        [ 0.3550, -0.3171,  0.2397,  0.7357,  0.3546,  0.0751, -0.0275,  0.4007],
        [ 0.3958, -0.2597, -0.0153,  0.5823,  0.1512,  0.0068, -0.0667,  0.4015],
        [ 0.2896, -0.1980, -0.2156,  0.1374,  0.1351, -0.0965, -0.1220,  0.1041]],
       grad_fn=<SelectBackward0>)


Causality test

In [49]:
#changing a future token must not change an earlier token's attention output.
# Make a copy so we don't change the original
modified_inputs = gpt_inputs.clone()

# Drastically change the final token at position 3
modified_inputs[:, 3, :] = 1000.0

# Run both through the SAME attention layer
original_output = mha_test(gpt_inputs)
modified_output = mha_test(modified_inputs)

print("Original outputs — Batch 0:")
print(original_output[0])

print("\nModified outputs — Batch 0:")
print(modified_output[0])

Original outputs — Batch 0:
tensor([[ 0.6649, -0.2722,  0.2148,  1.3442,  0.4512, -0.0245,  0.0608,  0.7016],
        [ 0.3550, -0.3171,  0.2397,  0.7357,  0.3546,  0.0751, -0.0275,  0.4007],
        [ 0.3958, -0.2597, -0.0153,  0.5823,  0.1512,  0.0068, -0.0667,  0.4015],
        [ 0.2896, -0.1980, -0.2156,  0.1374,  0.1351, -0.0965, -0.1220,  0.1041]],
       grad_fn=<SelectBackward0>)

Modified outputs — Batch 0:
tensor([[ 6.6495e-01, -2.7219e-01,  2.1485e-01,  1.3442e+00,  4.5124e-01,
         -2.4472e-02,  6.0783e-02,  7.0158e-01],
        [ 3.5502e-01, -3.1708e-01,  2.3966e-01,  7.3570e-01,  3.5464e-01,
          7.5058e-02, -2.7471e-02,  4.0066e-01],
        [ 3.9585e-01, -2.5966e-01, -1.5329e-02,  5.8227e-01,  1.5119e-01,
          6.8001e-03, -6.6707e-02,  4.0150e-01],
        [-2.2422e+02,  1.1000e+02, -2.4748e+02,  1.5880e+02,  2.2752e+02,
         -5.3304e+01, -3.2445e+02,  2.4552e+02]], grad_fn=<SelectBackward0>)


## Chapter 4: implementing the GPT model

In [50]:
#we have already constructed the multihead casual self attention
#Chapter 4 builds the surrounding Transformer machinery.
# The book's GPT implementation starts with the GPT-2-style architecture and then constructs the Transformer block and full GPT model.

In [51]:
#layer 4 normalization
#let's understand the layerNorm first
#take in the gpt_inputs

layer_norm = nn.LayerNorm(8)

normalized = layer_norm(gpt_inputs)

print("Input shape:", gpt_inputs.shape)
print("Normalized shape:", normalized.shape)

print("\nFirst token BEFORE LayerNorm:")
print(gpt_inputs[0, 0])

print("\nFirst token AFTER LayerNorm:")
print(normalized[0, 0])

print("\nMean after LayerNorm:")
print(normalized[0, 0].mean())

print("\nVariance after LayerNorm:")
print(normalized[0, 0].var(unbiased=False))

Input shape: torch.Size([2, 4, 8])
Normalized shape: torch.Size([2, 4, 8])

First token BEFORE LayerNorm:
tensor([ 0.8550,  1.5736,  0.8295,  0.2192,  0.3676, -2.3028, -2.8853,  0.8952],
       grad_fn=<SelectBackward0>)

First token AFTER LayerNorm:
tensor([ 0.5993,  1.0721,  0.5826,  0.1810,  0.2787, -1.4782, -1.8614,  0.6258],
       grad_fn=<SelectBackward0>)

Mean after LayerNorm:
tensor(2.2352e-08, grad_fn=<MeanBackward0>)

Variance after LayerNorm:
tensor(1.0000, grad_fn=<VarBackward0>)


In [52]:
print("Gamma (scale):")
print(layer_norm.weight)

print("\nBeta (shift):")
print(layer_norm.bias)

print("\nGamma shape:", layer_norm.weight.shape)
print("Beta shape:", layer_norm.bias.shape)

Gamma (scale):
Parameter containing:
tensor([1., 1., 1., 1., 1., 1., 1., 1.], requires_grad=True)

Beta (shift):
Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

Gamma shape: torch.Size([8])
Beta shape: torch.Size([8])


In [53]:
# #The Feed-Forward Network
# #second major computational component of a Transformer block.

# after attention has allowed "sat" to gather information from "The" and "cat",
# the feed-forward network further transforms the resulting representation of "sat".

In [54]:
#gelu
gelu = nn.GELU()

test_values = torch.tensor([
    -3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0
])

gelu_output = gelu(test_values)

print("Input:")
print(test_values)

print("\nAfter GELU:")
print(gelu_output)

Input:
tensor([-3., -2., -1.,  0.,  1.,  2.,  3.])

After GELU:
tensor([-0.0040, -0.0455, -0.1587,  0.0000,  0.8413,  1.9545,  2.9960])


In [55]:
#build the feedforward network
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        return self.layers(x)

In [56]:
#instantiate it for the got toy
ff = FeedForward(d_model=8)
print(ff)

FeedForward(
  (layers): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=32, out_features=8, bias=True)
  )
)


Why expand to 32 and then shrink back to 8?
The 32-dimensional internal space gives the network more capacity to transform the information, while returning to 8 dimensions is necessary because the rest of our Transformer operates with dmodel = 8

In [57]:
#Pass the GPT representations through the FFN
ff_output = ff(gpt_inputs)

print("Input shape:", gpt_inputs.shape)
print("FFN output shape:", ff_output.shape)

print("\nFirst token BEFORE FFN:")
print(gpt_inputs[0, 0])

print("\nFirst token AFTER FFN:")
print(ff_output[0, 0])

Input shape: torch.Size([2, 4, 8])
FFN output shape: torch.Size([2, 4, 8])

First token BEFORE FFN:
tensor([ 0.8550,  1.5736,  0.8295,  0.2192,  0.3676, -2.3028, -2.8853,  0.8952],
       grad_fn=<SelectBackward0>)

First token AFTER FFN:
tensor([-0.0536, -0.1235,  0.9407, -0.2010, -0.0550,  0.0328, -0.0301, -0.6998],
       grad_fn=<SelectBackward0>)


In [58]:
#Residual / Skip Connections
residual_output = gpt_inputs + ff_output

print("Original input shape:", gpt_inputs.shape)
print("FFN output shape:", ff_output.shape)
print("Residual output shape:", residual_output.shape)

print("\nOriginal first token:")
print(gpt_inputs[0, 0])

print("\nFFN transformation:")
print(ff_output[0, 0])

print("\nAfter residual connection:")
print(residual_output[0, 0])

Original input shape: torch.Size([2, 4, 8])
FFN output shape: torch.Size([2, 4, 8])
Residual output shape: torch.Size([2, 4, 8])

Original first token:
tensor([ 0.8550,  1.5736,  0.8295,  0.2192,  0.3676, -2.3028, -2.8853,  0.8952],
       grad_fn=<SelectBackward0>)

FFN transformation:
tensor([-0.0536, -0.1235,  0.9407, -0.2010, -0.0550,  0.0328, -0.0301, -0.6998],
       grad_fn=<SelectBackward0>)

After residual connection:
tensor([ 0.8013,  1.4501,  1.7703,  0.0182,  0.3126, -2.2701, -2.9154,  0.1954],
       grad_fn=<SelectBackward0>)


In [59]:
#build the transformer block

# Start with our GPT input
x = gpt_inputs

# ----- Attention part -----

# 1. Save x for the residual connection
shortcut = x

# 2. Normalize
x = layer_norm(x)

# 3. Causal multi-head attention
x = mha_test(x)

# 4. Add original representation back
x = x + shortcut

print("After attention + residual:")
print(x.shape)
print(x[0, 0])

After attention + residual:
torch.Size([2, 4, 8])
tensor([ 1.2883,  1.4035,  0.9689,  1.1170,  0.6798, -2.3176, -2.8517,  1.3559],
       grad_fn=<SelectBackward0>)


In [60]:
#Add the FFN half
# ----- Feed-forward part -----

# 1. Save the attention result for the residual connection
shortcut = x

# 2. Normalize
x = layer_norm(x)

# 3. Feed-forward network
x = ff(x)

# 4. Add the shortcut back
x = x + shortcut

print("After complete Transformer block:")
print(x.shape)

print("\nFirst token:")
print(x[0, 0])

After complete Transformer block:
torch.Size([2, 4, 8])

First token:
tensor([ 1.0832,  1.1574,  1.5479,  1.0024,  0.5730, -2.1472, -3.0063,  1.0211],
       grad_fn=<SelectBackward0>)


In [61]:
#Package it into TransformerBlock
class TransformerBlock(nn.Module):
    def __init__(self, d_model, context_length, num_heads):
        super().__init__()

        # Two separate normalization layers
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # Causal multi-head attention
        self.att = MultiHeadAttention(
            d_in=d_model,
            d_out=d_model,
            context_length=context_length,
            num_heads=num_heads
        )

        # Feed-forward network
        self.ff = FeedForward(d_model)

    def forward(self, x):

        # ----- Attention sub-block -----
        shortcut = x

        x = self.norm1(x)
        x = self.att(x)

        x = x + shortcut

        # ----- Feed-forward sub-block -----
        shortcut = x

        x = self.norm2(x)
        x = self.ff(x)

        x = x + shortcut

        return x

In [62]:
#instantiate it
transformer_block = TransformerBlock(
    d_model=8,
    context_length=4,
    num_heads=2
)

print(transformer_block)

TransformerBlock(
  (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (att): MultiHeadAttention(
    (W_query): Linear(in_features=8, out_features=8, bias=False)
    (W_key): Linear(in_features=8, out_features=8, bias=False)
    (W_value): Linear(in_features=8, out_features=8, bias=False)
    (out_proj): Linear(in_features=8, out_features=8, bias=False)
  )
  (ff): FeedForward(
    (layers): Sequential(
      (0): Linear(in_features=8, out_features=32, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=32, out_features=8, bias=True)
    )
  )
)


In [63]:
#Actually run the Transformer block
block_output = transformer_block(gpt_inputs)

print("Input shape:", gpt_inputs.shape)
print("Output shape:", block_output.shape)

print("\nFirst token BEFORE Transformer block:")
print(gpt_inputs[0, 0])

print("\nFirst token AFTER Transformer block:")
print(block_output[0, 0])

Input shape: torch.Size([2, 4, 8])
Output shape: torch.Size([2, 4, 8])

First token BEFORE Transformer block:
tensor([ 0.8550,  1.5736,  0.8295,  0.2192,  0.3676, -2.3028, -2.8853,  0.8952],
       grad_fn=<SelectBackward0>)

First token AFTER Transformer block:
tensor([ 1.5371,  1.7784,  0.7851,  0.1611,  1.1190, -2.3147, -2.7100,  1.1147],
       grad_fn=<SelectBackward0>)


In [64]:
#Stack multiple Transformer blocks
#Let's use 3 blocks in our toy GPT. Each block will have its own parameters.

In [65]:
num_layers = 3

transformer_blocks = nn.Sequential(
    *[
        TransformerBlock(
            d_model=8,
            context_length=4,
            num_heads=2
        )
        for _ in range(num_layers)
    ]
)

print(transformer_blocks)

Sequential(
  (0): TransformerBlock(
    (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
    (att): MultiHeadAttention(
      (W_query): Linear(in_features=8, out_features=8, bias=False)
      (W_key): Linear(in_features=8, out_features=8, bias=False)
      (W_value): Linear(in_features=8, out_features=8, bias=False)
      (out_proj): Linear(in_features=8, out_features=8, bias=False)
    )
    (ff): FeedForward(
      (layers): Sequential(
        (0): Linear(in_features=8, out_features=32, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=32, out_features=8, bias=True)
      )
    )
  )
  (1): TransformerBlock(
    (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
    (att): MultiHeadAttention(
      (W_query): Linear(in_features=8, out_features=8, bias=False)
      (W_key): Linear(in_features=8, out_fe

In [66]:
#Pass representations through all 3 blocks
stacked_output = transformer_blocks(gpt_inputs)

print("Input shape:", gpt_inputs.shape)
print("After 3 Transformer blocks:", stacked_output.shape)

print("\nFirst token BEFORE all blocks:")
print(gpt_inputs[0, 0])

print("\nFirst token AFTER all blocks:")
print(stacked_output[0, 0])

Input shape: torch.Size([2, 4, 8])
After 3 Transformer blocks: torch.Size([2, 4, 8])

First token BEFORE all blocks:
tensor([ 0.8550,  1.5736,  0.8295,  0.2192,  0.3676, -2.3028, -2.8853,  0.8952],
       grad_fn=<SelectBackward0>)

First token AFTER all blocks:
tensor([ 1.3516,  1.6621,  1.4283, -1.1178,  0.3617, -1.1178, -2.2689,  0.9635],
       grad_fn=<SelectBackward0>)


We currently have an 8-dimensional vector for each token. How does GPT turn that into a prediction like "the" or "cat" from a vocabulary of 50,257 tokens?

In [67]:
#Final LayerNorm
#Before projecting to vocabulary scores, GPT applies one final LayerNorm after the Transformer stack.
final_norm = nn.LayerNorm(8)

normalized_output = final_norm(stacked_output)

print("Before final norm:", stacked_output.shape)
print("After final norm:", normalized_output.shape)

print("\nFirst token after final norm:")
print(normalized_output[0, 0])

Before final norm: torch.Size([2, 4, 8])
After final norm: torch.Size([2, 4, 8])

First token after final norm:
tensor([ 0.8679,  1.0936,  0.9237, -0.9274,  0.1482, -0.9274, -1.7642,  0.5857],
       grad_fn=<SelectBackward0>)


In [68]:
#Output projection: representation → vocabulary logits
#Of all 50,257 possible GPT-2 tokens, which one should come next?

#create an output layer
output_head = nn.Linear(
    8,
    vocab_size,
    bias=False
)

logits = output_head(normalized_output)

print("Input to output head:", normalized_output.shape)
print("Output logits:", logits.shape)
print("\nVocabulary size:", vocab_size)


Input to output head: torch.Size([2, 4, 8])
Output logits: torch.Size([2, 4, 50257])

Vocabulary size: 50257


In [69]:
#Inspect logits and probabilities for one position
#Take the last position of the first sequence. This corresponds to "on":
# Logits for the final position of batch 0
last_position_logits = logits[0, 3]

print("Shape:", last_position_logits.shape)

print("\nFirst 10 logits:")
print(last_position_logits[:10])

print("\nCorrect target token ID:")
print(targets[0, 3])

print("\nCorrect target token:")
print(repr(tokenizer.decode([targets[0, 3].item()])))

Shape: torch.Size([50257])

First 10 logits:
tensor([ 0.3755,  0.4684,  0.2743,  0.5237,  0.0379, -0.3683,  0.2171,  1.1399,
        -0.2662, -0.8795], grad_fn=<SliceBackward0>)

Correct target token ID:
tensor(262)

Correct target token:
' the'


In [70]:
probabilities = torch.softmax(last_position_logits, dim=-1)

print("Probabilities shape:", probabilities.shape)
print("Sum of probabilities:", probabilities.sum())

correct_token_id = targets[0, 3].item()

print("\nCorrect token:", repr(tokenizer.decode([correct_token_id])))
print("Correct token ID:", correct_token_id)

print("\nProbability assigned to correct token:")
print(probabilities[correct_token_id])

Probabilities shape: torch.Size([50257])
Sum of probabilities: tensor(1.0000, grad_fn=<SumBackward0>)

Correct token: ' the'
Correct token ID: 262

Probability assigned to correct token:
tensor(1.6101e-05, grad_fn=<SelectBackward0>)


In [71]:
#Build the complete GPTModel
class GPTModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        context_length,
        num_heads,
        num_layers
    ):
        super().__init__()

        # 1. Token embeddings
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # 2. Position embeddings
        self.position_embedding = nn.Embedding(
            context_length,
            d_model
        )

        # 3. Stack of Transformer blocks
        self.transformer_blocks = nn.Sequential(
            *[
                TransformerBlock(
                    d_model=d_model,
                    context_length=context_length,
                    num_heads=num_heads
                )
                for _ in range(num_layers)
            ]
        )

        # 4. Final normalization
        self.final_norm = nn.LayerNorm(d_model)

        # 5. Vocabulary output projection
        self.output_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

    def forward(self, token_ids):

        B, T = token_ids.shape

        # Token embeddings
        token_emb = self.token_embedding(token_ids)

        # Position IDs: [0, 1, 2, ..., T-1]
        positions = torch.arange(
            T,
            device=token_ids.device
        )

        # Position embeddings
        pos_emb = self.position_embedding(positions)

        # Combine token + position information
        x = token_emb + pos_emb

        # Transformer blocks
        x = self.transformer_blocks(x)

        # Final LayerNorm
        x = self.final_norm(x)

        # Produce one score for every vocabulary token
        logits = self.output_head(x)

        return logits

In [72]:
#instantiate
model = GPTModel(
    vocab_size=vocab_size,
    d_model=8,
    context_length=4,
    num_heads=2,
    num_layers=3
)

print(model)

GPTModel(
  (token_embedding): Embedding(50257, 8)
  (position_embedding): Embedding(4, 8)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=8, out_features=8, bias=False)
        (W_key): Linear(in_features=8, out_features=8, bias=False)
        (W_value): Linear(in_features=8, out_features=8, bias=False)
        (out_proj): Linear(in_features=8, out_features=8, bias=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=8, out_features=32, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=32, out_features=8, bias=True)
        )
      )
    )
    (1): TransformerBlock(
      (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((8,), eps=1e-05, elementwise

In [73]:
#run the token IDS through it
logits = model(inputs)

print("Input token IDs shape:")
print(inputs.shape)

print("\nGPT output logits shape:")
print(logits.shape)

Input token IDs shape:
torch.Size([2, 4])

GPT output logits shape:
torch.Size([2, 4, 50257])


In [74]:
#Calculate the loss
import torch.nn.functional as F

logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("Original logits:", logits.shape)
print("Flattened logits:", logits_flat.shape)

print("\nOriginal targets:", targets.shape)
print("Flattened targets:", targets_flat.shape)

Original logits: torch.Size([2, 4, 50257])
Flattened logits: torch.Size([8, 50257])

Original targets: torch.Size([2, 4])
Flattened targets: torch.Size([8])


In [75]:
#calculate cross entropy loss
loss = F.cross_entropy(
    logits_flat,
    targets_flat
)

print("Cross-entropy loss:")
print(loss)

Cross-entropy loss:
tensor(10.7425, grad_fn=<NllLossBackward0>)


In [76]:
#Backpropagation
id="4zcn6y"
param = model.transformer_blocks[0].att.W_query.weight

print("W_query gradient BEFORE backward:")
print(param.grad)

W_query gradient BEFORE backward:
None


In [77]:
id="7c8h6m"
loss.backward()

In [78]:
id="4hwtkg"
print("W_query gradient AFTER backward:")
print(param.grad)

print("\nGradient shape:")
print(param.grad.shape)

W_query gradient AFTER backward:
tensor([[ 1.8348e-03,  3.8902e-03, -3.1218e-03, -4.9533e-03,  2.6023e-03,
         -5.1947e-03,  1.0940e-02, -5.9971e-03],
        [-1.9068e-05,  1.9002e-03,  1.2891e-03, -1.6261e-04, -3.0386e-04,
          2.0444e-03, -5.1281e-03,  3.7996e-04],
        [ 2.1469e-03,  5.2884e-03, -3.8664e-03, -6.9728e-03,  3.5723e-03,
         -7.0433e-03,  1.5490e-02, -8.6149e-03],
        [-1.2115e-03, -4.4599e-03,  2.0889e-03,  3.9167e-03, -1.8283e-03,
          3.5145e-03, -6.6754e-03,  4.6550e-03],
        [-3.3302e-03, -7.0221e-04,  2.2689e-03,  3.0773e-03, -2.3257e-03,
          2.6382e-03, -2.5713e-03,  9.4498e-04],
        [-1.9218e-03,  3.4540e-04,  1.2065e-03,  1.1441e-03, -1.0417e-03,
          1.0522e-03, -6.5449e-04, -1.3020e-04],
        [ 4.1912e-03,  3.3301e-04, -2.3834e-03, -1.1486e-03,  1.2311e-03,
         -4.9159e-04, -3.3290e-03,  1.5972e-03],
        [ 2.6124e-04, -1.1215e-03,  4.6979e-04,  1.8900e-03, -9.8304e-04,
          2.0346e-03, -5.1244e-0

In [79]:
#Let the adamW update the weights
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

In [80]:
optimizer.zero_grad()

In [81]:
# 1. Forward pass
logits = model(inputs)

# 2. Calculate loss
loss = F.cross_entropy(
    logits.flatten(0, 1),
    targets.flatten()
)

# 3. Backpropagation
loss.backward()

# 4. Update model parameters
optimizer.step()

print("Loss:", loss.item())

Loss: 10.742542266845703


In [82]:
#Check the loss after one update
# Forward pass again AFTER the optimizer updated the weights
with torch.no_grad():
    new_logits = model(inputs)

    new_loss = F.cross_entropy(
        new_logits.flatten(0, 1),
        targets.flatten()
    )

print("Loss before update:", loss.item())
print("Loss after update: ", new_loss.item())

Loss before update: 10.742542266845703
Loss after update:  10.703584671020508


In [83]:
#Train for multiple steps
num_steps = 100

for step in range(num_steps):

    # 1. Clear gradients from previous step
    optimizer.zero_grad()

    # 2. Forward pass
    logits = model(inputs)

    # 3. Calculate next-token prediction loss
    loss = F.cross_entropy(
        logits.flatten(0, 1),
        targets.flatten()
    )

    # 4. Backpropagation
    loss.backward()

    # 5. Update parameters
    optimizer.step()

    # Print occasionally
    if step % 10 == 0:
        print(f"Step {step:3d} | Loss: {loss.item():.4f}")

Step   0 | Loss: 10.7036
Step  10 | Loss: 10.3531
Step  20 | Loss: 10.0638
Step  30 | Loss: 9.8131
Step  40 | Loss: 9.5876
Step  50 | Loss: 9.3761
Step  60 | Loss: 9.1702
Step  70 | Loss: 8.9651
Step  80 | Loss: 8.7568
Step  90 | Loss: 8.5416


In [84]:
#Inspect the model's predictions
with torch.no_grad():
    logits = model(inputs)

# Highest-scoring vocabulary token at every position
predicted_ids = torch.argmax(logits, dim=-1)

print("Input IDs:")
print(inputs)

print("\nTarget IDs:")
print(targets)

print("\nPredicted IDs:")
print(predicted_ids)

Input IDs:
tensor([[ 464, 3797, 3332,  319],
        [3797, 3332,  319,  262]])

Target IDs:
tensor([[3797, 3332,  319,  262],
        [3332,  319,  262, 2603]])

Predicted IDs:
tensor([[3797, 3332,  319,  262],
        [3332,  319,  319, 2603]])


In [85]:
print("First sequence:\n")

for i in range(inputs.shape[1]):

    input_token = tokenizer.decode([inputs[0, i].item()])
    target_token = tokenizer.decode([targets[0, i].item()])
    predicted_token = tokenizer.decode([predicted_ids[0, i].item()])

    print(
        f"Input: {repr(input_token):12s} "
        f"Target: {repr(target_token):12s} "
        f"Prediction: {repr(predicted_token)}"
    )

First sequence:

Input: 'The'        Target: ' cat'       Prediction: ' cat'
Input: ' cat'       Target: ' sat'       Prediction: ' sat'
Input: ' sat'       Target: ' on'        Prediction: ' on'
Input: ' on'        Target: ' the'       Prediction: ' the'


In [86]:
with torch.no_grad():
    logits = model(inputs)
    probabilities = torch.softmax(logits, dim=-1)

print("First sequence:\n")

for i in range(inputs.shape[1]):

    target_id = targets[0, i].item()

    target_token = tokenizer.decode([target_id])

    probability = probabilities[0, i, target_id].item()

    print(
        f"Target: {repr(target_token):10s} "
        f"Probability: {probability:.6f}"
    )

First sequence:

Target: ' cat'     Probability: 0.000361
Target: ' sat'     Probability: 0.000246
Target: ' on'      Probability: 0.000351
Target: ' the'     Probability: 0.000146


In [87]:
#Train longer
for step in range(500):

    optimizer.zero_grad()

    logits = model(inputs)

    loss = F.cross_entropy(
        logits.flatten(0, 1),
        targets.flatten()
    )

    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"Step {step:3d} | Loss: {loss.item():.4f}")

Step   0 | Loss: 8.3188
Step  50 | Loss: 7.0526
Step 100 | Loss: 5.4407
Step 150 | Loss: 3.5202
Step 200 | Loss: 1.8924
Step 250 | Loss: 1.0443
Step 300 | Loss: 0.6020
Step 350 | Loss: 0.3722
Step 400 | Loss: 0.2515
Step 450 | Loss: 0.1828


In [88]:
#Check confidence again
with torch.no_grad():
    logits = model(inputs)
    probabilities = torch.softmax(logits, dim=-1)

print("First sequence:\n")

for i in range(inputs.shape[1]):

    target_id = targets[0, i].item()
    target_token = tokenizer.decode([target_id])

    probability = probabilities[0, i, target_id].item()

    print(
        f"Target: {repr(target_token):10s} "
        f"Probability: {probability:.6f}"
    )

First sequence:

Target: ' cat'     Probability: 0.808525
Target: ' sat'     Probability: 0.857061
Target: ' on'      Probability: 0.898903
Target: ' the'     Probability: 0.897894


In [89]:
#Give GPT only a prompt

prompt = "The"

prompt_ids = tokenizer.encode(prompt)

print("Prompt:", prompt)
print("Token IDs:", prompt_ids)
print("Decoded:", tokenizer.decode(prompt_ids))

Prompt: The
Token IDs: [464]
Decoded: The


In [90]:
#FIRST CONVERT THE PROMPT INTO A [BATCH, SEQUENCE] SHAPE THAT THE MODEL EXPECTS
input_ids = torch.tensor(prompt_ids).unsqueeze(0)

print("Input IDs:")
print(input_ids)

print("Shape:")
print(input_ids.shape)

Input IDs:
tensor([[464]])
Shape:
torch.Size([1, 1])


In [91]:
#FEED THE TOKEN TO GPT
with torch.no_grad():
    logits = model(input_ids)

print("Logits shape:")
print(logits.shape)

Logits shape:
torch.Size([1, 1, 50257])


In [93]:
next_token_logits = logits[:, -1, :]

print("Next-token logits shape:")
print(next_token_logits.shape)

Next-token logits shape:
torch.Size([1, 50257])


In [94]:
#Choose the highest-scoring next token
next_token_id = torch.argmax(
    next_token_logits,
    dim=-1
)

print("Predicted token ID:")
print(next_token_id)

print("\nPredicted token:")
print(repr(tokenizer.decode(next_token_id.tolist())))

Predicted token ID:
tensor([3797])

Predicted token:
' cat'


In [95]:
#Append the prediction to the context
input_ids = torch.cat(
    (input_ids, next_token_id.unsqueeze(0)),
    dim=1
)

print("Updated token IDs:")
print(input_ids)

print("\nUpdated text:")
print(repr(tokenizer.decode(input_ids[0].tolist())))

print("\nShape:")
print(input_ids.shape)

Updated token IDs:
tensor([[ 464, 3797]])

Updated text:
'The cat'

Shape:
torch.Size([1, 2])


In [96]:
#Feed "The cat" back into GPT
with torch.no_grad():
    logits = model(input_ids)

print("Logits shape:")
print(logits.shape)

Logits shape:
torch.Size([1, 2, 50257])


In [97]:
next_token_logits = logits[:, -1, :]

next_token_id = torch.argmax(
    next_token_logits,
    dim=-1
)

print("Predicted token ID:")
print(next_token_id)

print("\nPredicted token:")
print(repr(tokenizer.decode(next_token_id.tolist())))

Predicted token ID:
tensor([3332])

Predicted token:
' sat'


In [98]:
#Autoregressive generation loop
input_ids = torch.cat(
    (input_ids, next_token_id.unsqueeze(0)),
    dim=1
)

print(tokenizer.decode(input_ids[0].tolist()))

The cat sat


In [99]:
#FUNCTION
def generate_text_simple(model, input_ids, max_new_tokens, context_length):

    for _ in range(max_new_tokens):

        # Keep only tokens that fit inside our context window
        input_cond = input_ids[:, -context_length:]

        # Forward pass
        with torch.no_grad():
            logits = model(input_cond)

        # Only the final position predicts the next token
        logits = logits[:, -1, :]

        # Greedy decoding
        next_token_id = torch.argmax(logits, dim=-1, keepdim=True)

        # Append prediction to sequence
        input_ids = torch.cat(
            (input_ids, next_token_id),
            dim=1
        )

    return input_ids

In [100]:
prompt = "The"

input_ids = torch.tensor(
    tokenizer.encode(prompt)
).unsqueeze(0)

print("Starting input:")
print(input_ids)

print("\nStarting text:")
print(tokenizer.decode(input_ids[0].tolist()))

Starting input:
tensor([[464]])

Starting text:
The


In [101]:
generated_ids = generate_text_simple(
    model=model,
    input_ids=input_ids,
    max_new_tokens=3,
    context_length=4
)

generated_text = tokenizer.decode(
    generated_ids[0].tolist()
)

print("Generated token IDs:")
print(generated_ids)

print("\nGenerated text:")
print(repr(generated_text))

Generated token IDs:
tensor([[ 464, 3797, 3332,  319]])

Generated text:
'The cat sat on'


In [102]:
#Generate beyond the context window
prompt = "The"

input_ids = torch.tensor(
    tokenizer.encode(prompt)
).unsqueeze(0)

generated_ids = generate_text_simple(
    model=model,
    input_ids=input_ids,
    max_new_tokens=8,
    context_length=4
)

print("Generated IDs:")
print(generated_ids)

print("\nGenerated text:")
print(repr(tokenizer.decode(generated_ids[0].tolist())))

Generated IDs:
tensor([[ 464, 3797, 3332,  319,  262, 2603, 2603, 2603, 2603]])

Generated text:
'The cat sat on the mat mat mat mat'


In [103]:
#Fix the training data first
for batch_idx, (batch_inputs, batch_targets) in enumerate(dataloader):

    print(f"\nBatch {batch_idx}")

    for i in range(batch_inputs.shape[0]):
        input_text = tokenizer.decode(batch_inputs[i].tolist())
        target_text = tokenizer.decode(batch_targets[i].tolist())

        print(
            f"Input:  {repr(input_text):30s} "
            f"Target: {repr(target_text)}"
        )


Batch 0
Input:  'The cat sat on'               Target: ' cat sat on the'
Input:  ' cat sat on the'              Target: ' sat on the mat'

Batch 1
Input:  ' sat on the mat'              Target: ' on the mat because'
Input:  ' on the mat because'          Target: ' the mat because the'

Batch 2
Input:  ' the mat because the'         Target: ' mat because the cat'
Input:  ' mat because the cat'         Target: ' because the cat was'

Batch 3
Input:  ' because the cat was'         Target: ' the cat was tired'
Input:  ' the cat was tired'           Target: ' cat was tired.'


In [104]:
#TRAIN FOR ONE EPOCH
model.train()

for batch_idx, (batch_inputs, batch_targets) in enumerate(dataloader):

    # 1. Clear previous gradients
    optimizer.zero_grad()

    # 2. Forward pass
    logits = model(batch_inputs)

    # 3. Calculate loss
    loss = F.cross_entropy(
        logits.flatten(0, 1),
        batch_targets.flatten()
    )

    # 4. Backpropagation
    loss.backward()

    # 5. Update parameters
    optimizer.step()

    print(
        f"Batch {batch_idx} | "
        f"Loss: {loss.item():.4f}"
    )

Batch 0 | Loss: 0.1401
Batch 1 | Loss: 5.1355
Batch 2 | Loss: 7.2435
Batch 3 | Loss: 9.8538


In [105]:
#PROPER MULTI EPOCH TRAINING

num_epochs = 100

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for batch_inputs, batch_targets in dataloader:

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(batch_inputs)

        # Loss
        loss = F.cross_entropy(
            logits.flatten(0, 1),
            batch_targets.flatten()
        )

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(dataloader)

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Average loss: {average_loss:.4f}"
        )

Epoch   0 | Average loss: 4.9072
Epoch  10 | Average loss: 3.1809
Epoch  20 | Average loss: 2.4609
Epoch  30 | Average loss: 1.9946
Epoch  40 | Average loss: 1.5946
Epoch  50 | Average loss: 1.2160
Epoch  60 | Average loss: 0.8736
Epoch  70 | Average loss: 0.6088
Epoch  80 | Average loss: 0.4359
Epoch  90 | Average loss: 0.3271


In [106]:
#GENERATE AGAIN
prompt = "The"

input_ids = torch.tensor(
    tokenizer.encode(prompt)
).unsqueeze(0)

generated_ids = generate_text_simple(
    model=model,
    input_ids=input_ids,
    max_new_tokens=11,
    context_length=4
)

generated_text = tokenizer.decode(
    generated_ids[0].tolist()
)

print("Generated IDs:")
print(generated_ids)

print("\nGenerated text:")
print(repr(generated_text))

Generated IDs:
tensor([[  464,  3797,  3332,   319,   262,  2603,   780,   262,  3797,   373,
         10032,    13]])

Generated text:
'The cat sat on the mat because the cat was tired.'


In [107]:
#TEST
prompt = "A dog"

input_ids = torch.tensor(
    tokenizer.encode(prompt)
).unsqueeze(0)

generated_ids = generate_text_simple(
    model=model,
    input_ids=input_ids,
    max_new_tokens=8,
    context_length=4
)

print(
    repr(
        tokenizer.decode(
            generated_ids[0].tolist()
        )
    )
)

'A dog the cat was tired. because the cat'
